In [1]:
# 언어 모델 선언
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage(content="좋은 아침이야. 공부에 집중되게 빠이팅해줘")])

AIMessage(content='좋은 아침이에요! 오늘도 공부 열심히 해서 원하는 목표를 이뤄내길 응원할게요. 힘내고, 집중해서 좋은 결과 얻길 바라요! 파이팅! 💪📚', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 24, 'total_tokens': 73, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-Bzvv2VX3sHc5tlAixhqW0vP7vdPGT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--fcddd5a2-e76e-413d-b0a4-7325007fd636-0', usage_metadata={'input_tokens': 24, 'output_tokens': 49, 'total_tokens': 73, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하면 함수를 툴로 등록할 수 있음
def get_current_time(timezone: str = "Asia/Seoul", location: str = "서울") -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (기본값: "Asia/Seoul"). 실재 존재해야 함
        location (str): 지역명. 타임존은 모든 지명에 대응되지 않으므로 이후 llm 답변 생성에 사용됨
    """

    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    localtion_and_local_time = f"{location}({timezone})의 현재 시각은 {now} 입니다."

    print(localtion_and_local_time)
    return localtion_and_local_time

In [3]:
# 함수 get_current_time 을 랭체인으로 llm에 연결하기
tools = [get_current_time]
tool_dict = {"get_current_time": get_current_time}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [16]:
# 사용자의 질문과 도구를 사용해 언어 모델 답변 생성하기
from langchain_core.messages import SystemMessage

# 사용자의 질문과 도구를 사용해 언어 모델 답변 생성
messages = [
    SystemMessage(content="너는 사용자의 질문에 답변을 하기 위해 tools 를 사용할 수 있다."),
    HumanMessage(content="부산의 현재 시각은 몇 시야?")
]

# llm_with_tools 를 사용해 사용자의 질문에 언어 모델 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# 생성된 언어 모델 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools 를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산의 현재 시각은 몇 시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_2MJuaKcDWCI87pjA4sdu8BAf', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"부산"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 152, 'total_tokens': 175, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BzwHA1c4kDsjoXFzKR5aNf33JSduY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--5dd1de65-4af1-40dd-8c22-1c5e272c97bb-0', tool

In [17]:
# 함수 싦행 결과 출력하기
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call['name']] # tool_dict를 사용해 도구 함수 선택
    print(tool_call['args']) # 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # 도구 함수를 호출한 결과 반환
    messages.append(tool_msg) # 도구 함수 결과를 메시지 리스트에 추가

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
부산(Asia/Seoul)의 현재 시각은 2025-08-02 11:28:42 입니다.


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools 를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산의 현재 시각은 몇 시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_2MJuaKcDWCI87pjA4sdu8BAf', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"부산"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 152, 'total_tokens': 175, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BzwHA1c4kDsjoXFzKR5aNf33JSduY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--5dd1de65-4af1-40dd-8c22-1c5e272c97bb-0', to

In [18]:
# 함수 실행 결과를 문장으로 출력하기
llm_with_tools.invoke(messages)

AIMessage(content='부산의 현재 시각은 2025년 8월 2일 11시 28분 42초입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 211, 'total_tokens': 240, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BzwHPOmSC2YT5Bl9AwK9USUMI1upE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--625af71d-8f34-441d-937d-fcc911394f09-0', usage_metadata={'input_tokens': 211, 'output_tokens': 29, 'total_tokens': 240, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [19]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="종목 코드", description="주식 종목 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="기간 (예: 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max)")

In [20]:
import yfinance as yf

@tool
def get_yf_stock_history(StockHistoryInput: StockHistoryInput) -> str:
    """주식 종목의 가격 데이터를 조회하는 함수"""

    stock = yf.Ticker(StockHistoryInput.ticker)
    history = stock.history(period=StockHistoryInput.period)
    history_md = history.to_markdown()

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [21]:
messages.append(HumanMessage(content="테슬라는 한 달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'tool_calls': [{'id': 'call_0QMD7VU53J9Dt3j45E8ntDD1', 'function': {'arguments': '{"StockHistoryInput":{"ticker":"TSLA","period":"1mo"}}', 'name': 'get_yf_stock_history'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 329, 'total_tokens': 356, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BzxR7TWlLQ1KDhIIYiVdnZcsm2nOG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--daaaa4ed-7dce-481e-a267-b113c4ceed20-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'StockHistoryInput': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_0QMD7VU53J9Dt3j45E8ntDD1', 'type': 'tool_call'}] usage_metadata={'input_t

In [22]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call['name']]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'StockHistoryInput': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2025-07-02 00:00:00-04:00 | 312.63 | 316.83 | 303.82 |  315.65 | 1.19484e+08 |           0 |              0 |\n| 2025-07-03 00:00:00-04:00 | 317.99 | 318.45 | 312.76 |  315.35 | 5.80423e+07 |           0 |              0 |\n| 2025-07-07 00:00:00-04:00 | 291.37 | 296.15 | 288.77 |  293.94 | 1.31178e+08 |           0 |              0 |\n| 2025-07-08 00:00:00-04:00 | 297    | 304.05 | 294.35 |  297.81 | 1.03247e+08 |           0 |              0 |\n| 2025-07-09 00:00:00-04:00 | 297.55 | 300.15 | 293.55 |  295.88 | 7.55868e+07 |           0 |              0 |\n| 2025-07-10 00:00:00-04:00 | 300.05 | 310.48 | 300    |  309.87 | 1.04365e+08 |           0 |              0 |\n| 2025-07-11 00:00:00-04:00 

In [23]:
llm_with_tools.invoke(messages)

AIMessage(content='테슬라(TSLA)의 주가 변동을 살펴보면, 한 달 전인 2025년 7월 2일의 종가가 **315.65** 달러였습니다. 현재의 종가는 2025년 8월 1일에 **302.63** 달러였으므로, 한 달 동안 주가는 내렸습니다. \n\n정리하면, 테슬라의 주가는 한 달 전보다 하락하였습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 1686, 'total_tokens': 1786, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BzxSqbK4bdcvlj4321Gzn9fD55v6K', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--1a7846c5-cb37-438f-b969-e895ef1158de-0', usage_metadata={'input_tokens': 1686, 'output_tokens': 100, 'total_tokens': 1786, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [24]:
# 랭체인 도구 사용 시 스트리밍 방식으로 출력하기
for c in llm.stream([HumanMessage(content="잘 지냈어? 한국 사회의 문제점이 무엇인지 이야기해줘.")]):
    print(c.content, end="|")

|안|녕하세요|!| 한국| 사회|의| 문제|점|은| 여러| 가지|가| 있으며|,| 그| 중| 몇| 가지|를| 소개|하|겠습니다|.

|1|.| **|고|용| 불|안|정|**|:| 청|년|층|을| 비롯|한| 많은| 사람들이| 안정|적인| 일|자|리를| 찾|기| 어려|워|하고| 있습니다|.| 비|정|규|직|이나| 프|리|랜|서|로| 일|하는| 경우|가| 많|고|,| 이|로| 인해| 경제|적| 불|안|정|성이| 커|지고| 있습니다|.

|2|.| **|성|별| 불|평|등|**|:| 여성|의| 경제|적| 지|위|가| 상승|하고| 있지만|,| 여|전히| 성|별|에| 따른| 임|금| 격|차|와| 직|장| 내| 성|희|롱| 등의| 문제|는| 해결|되지| 않고| 있습니다|.| 직|장| 내| 유|리|천|장|도| 여|전히| 존재|합니다|.

|3|.| **|고|령|화| 사회|**|:| 한국|은| 빠|르게| 고|령|화|가| 진행|되고| 있으며|,| 이에| 따른| 사회|적| 요구|와| 복|지| 시스템|의| 부족|이| 문제|로| 지|적|되고| 있습니다|.| 노|인| 빈|곤|율|이| 높|아|지고| 있어|,| 노|인| 복|지| 정책|의| 강화|가| 필요|합니다|.

|4|.| **|주|거| 문제|**|:| 주|택| 가격|의| 상승|과| 전|세|난| 등|으로| 인해| 많은| 사람들이| 주|거| 문제|로| 어려|움을| 겪|고| 있습니다|.| 특히| 청|년|층|은| 주|택| 마련|이| 어려|워|지고| 있습니다|.

|5|.| **|정|신| 건강| 문제|**|:| 정신| 건강|에| 대한| 인|식|이| 점|차| 개선|되고| 있지만| 여|전히| stigma|가| 존재|합니다|.| 우|울|증|,| 불|안|증| 등의| 정신| 건강| 문제가| 심|각|한| 상황|에서도| 적|절|한| 지원|을| 받|기| 어려|운| 경우|가| 많|습니다|.

|6|.| **|교육| 경쟁|**|:| 한국|의| 교육| 시스템|은| 학생|들에게| 큰| 부담|을| 주|고| 있으며|,| 지나|친| 경쟁|과| 사|교육| 의|존|도가|

In [26]:
# 도구 호출(tool call) 청크 모아서 하나로 만들기

messages = [
    SystemMessage(content="너는 사용자의 질문에 답변을 하기 위해 tools 를 사용할 수 있다."),
    HumanMessage(content="테슬라는 한 달 전에 비해 주가가 올랐나 내렸나?")
]

response = llm_with_tools.stream(messages)

is_first = True

for chunk in response:
    print('chunk type: ', type(chunk))
    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk

    print("content: ", gathered.content, "tool_call_chunks: ", gathered.tool_calls)

messages.append(gathered)

chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunks:  [{'name': 'get_yf_stock_history', 'args': {}, 'id': 'call_xLmVlUe9yAp9eXRCuBP2TTJk', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunks:  [{'name': 'get_yf_stock_history', 'args': {}, 'id': 'call_xLmVlUe9yAp9eXRCuBP2TTJk', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunks:  [{'name': 'get_yf_stock_history', 'args': {}, 'id': 'call_xLmVlUe9yAp9eXRCuBP2TTJk', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunks:  [{'name': 'get_yf_stock_history', 'args': {}, 'id': 'call_xLmVlUe9yAp9eXRCuBP2TTJk', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunks:  [{'name': 'get_yf_stock_history', 'args': {}, 'id': 'call_xLmVlUe9yAp9eXRCuBP2TTJk', 'type': 'too

In [27]:
gathered

AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_xLmVlUe9yAp9eXRCuBP2TTJk', 'function': {'arguments': '{"StockHistoryInput":{"ticker":"TSLA","period":"1mo"}}', 'name': 'get_yf_stock_history'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'service_tier': 'default'}, id='run--cff90174-58a4-405a-a11f-f753f6673462', tool_calls=[{'name': 'get_yf_stock_history', 'args': {'StockHistoryInput': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_xLmVlUe9yAp9eXRCuBP2TTJk', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'get_yf_stock_history', 'args': '{"StockHistoryInput":{"ticker":"TSLA","period":"1mo"}}', 'id': 'call_xLmVlUe9yAp9eXRCuBP2TTJk', 'index': 0, 'type': 'tool_call_chunk'}])

In [ ]:
# AIMessageChunk에 기존 대화를 추가해서 도구 사용하기
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call['name']] # tool_dict 를 사용하여 도구 이름으로 도구 함수 선택
    print(tool_call["args"] )
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

{'StockHistoryInput': {'ticker': 'TSLA', 'period': '1mo'}}
content='테슬라(TSLA)의 주가는 한 달 전과 비교했을 때 다음과 같습니다:\n\n- 한 달 전 주가 (2025년 07월 01일 기준): 약 **306.21**\n- 현재 주가 (2025년 08월 01일 기준): 약 **302.63**\n\n결과적으로, 테슬라의 주가는 한 달 전에 비해 **내렸습니다.**' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 1612, 'total_tokens': 1701, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BzxmNmbZ6TYpmrkisYmeepNFFm9HG', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--5cac70b0-eef7-40bd-8410-9d59456a001d-0' usage_metadata={'input_tokens': 1612, 'output_tokens': 89, 'total_tokens': 1701, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0

In [29]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools 를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='테슬라는 한 달 전에 비해 주가가 올랐나 내렸나?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_xLmVlUe9yAp9eXRCuBP2TTJk', 'function': {'arguments': '{"StockHistoryInput":{"ticker":"TSLA","period":"1mo"}}', 'name': 'get_yf_stock_history'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'service_tier': 'default'}, id='run--cff90174-58a4-405a-a11f-f753f6673462', tool_calls=[{'name': 'get_yf_stock_history', 'args': {'StockHistoryInput': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_xLmVlUe9yAp9eXRCuBP2TTJk', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'get_yf_stock_history', 'args': '{"StockHistoryInput":{"ticker":"TSLA","period":"1mo"}}', 'id': 'call_xLmVlUe9yAp9eXRCuBP2TTJk', '

In [30]:
# tool 정보 llm 에게 보내서 도구 사용해 스트림 방식 출력하기
for c in llm_with_tools.stream(messages):
    print(c.content, end="|")

|한| 달| 전| 테|슬|라|(|Stock| T|icker|:| TS|LA|)의| 주|가는| |202|5|년| |7|월| |1|일| 기준|으로| 약| |306|.|21| 달|러|였|고|,| 현재| 주|가는| |202|5|년| |8|월| |1|일| 기준|으로| 약| |302|.|63| 달|러|입니다|.

|따|라|서| 한| 달| 전|과| 비교|했|을| 때| 테|슬|라|의| 주|가는| **|내|렸|습니다|**|.||